# CampusRSVP: run in Google Colab / Jupyter
This notebook installs the dependencies, fetches the project and launches the Flask app.
Edit `REPO_URL` below to point at your GitHub repository.

In [ ]:
REPO_URL = 'https://github.com/<your-username>/campus-rsvp.git'  # <- edit this
!git clone $REPO_URL campus-rsvp 2>/dev/null || echo 'already cloned'
%cd campus-rsvp
!pip install -q flask flask-sqlalchemy pyngrok pytest

## 1. Run the unit tests

In [ ]:
!python -m pytest -q -p no:warnings

## 2. Seed the database and start the server
In Colab the app is exposed through the built-in proxy (a public link is printed). In a local Jupyter just open http://127.0.0.1:5000.

In [ ]:
!python seed.py
import threading, time, os
from app import app
threading.Thread(target=lambda: app.run(port=5000, use_reloader=False), daemon=True).start()
time.sleep(2)
try:
    from google.colab import output
    output.serve_kernel_port_as_window(5000)   # opens the running site in a new tab
except ImportError:
    print('Running locally: open http://127.0.0.1:5000')

## 3. Peek at the backend
Query the REST endpoint and the SQLite tables directly.

In [ ]:
import requests, sqlite3, json
print(json.dumps(requests.get('http://127.0.0.1:5000/api/events').json()[:2], indent=2))
print(sqlite3.connect('instance/campus.db').execute('select count(*) from rsvp').fetchone())